#### **Problem Statement:**

- Prompt Engineering is the technique of structuring text inputs - that can be interpreted and understood by generative AI models.

- Prompt Engineering involves developing and optimizing prompts to efficiently utliize large language models (LLM) for various applications.

- Some examples of Large Language Models are : Gemini, BERT, GPT-3, Llama, etc.

- Large refers to the number of parameters in the model. For example : BERT has 110M parameters, GPT-3 has 175B parameters and so on.

#### **Best Practices in Prompt Engineering:**

- Being as specific and clear as possible.
- Structure the prompt in a way to provide the context around the question.
- Use specific examples to help the model provide accurate responses.
- Use of one-shot or few-shot prompting techniques.
- Iterate and refine based on the response received from the large language model.

#### **References**

- https://www.promptingguide.ai/

- https://huggingface.co/docs

- Ekin, Sabit. "Prompt engineering for ChatGPT: A quick guide to techniques, tips, and best practices." Authorea Preprints (2023).

- Hadi, Muhammad Usman, et al. "A survey on large language models: Applications, challenges, limitations, and practical usage." Authorea Preprints (2023).

- https://ai.google.dev/examples/doc_search_emb

#### **Installing dependencies**

In [ ]:
# Installing google-generativeai library.

# Pinned versions (tested Oct 2026) so this notebook keeps working on Google Colab
!pip install -q google-generativeai==0.8.5
# !pip install -q -U google-generativeai   # (replaced by the pinned install above)

#### **Importing required pacakages**

In [ ]:
# Importing dependencies to work with the Gemini language model
# Importing genai from google.generativeai
# Importing userdata from google.colab
# Import libraries for file and text manipulation
# Importing Markdown and display from Ipython.display for formatting response from model

import pathlib
import textwrap

import pandas as pd
import numpy as np

import google.generativeai as genai
import google.ai.generativelanguage as glm

# Used to securely store API Key
from google.colab import userdata

from IPython.display import display
from IPython.display import Markdown

#### **Setup API Key**

In [ ]:
# https://makersuite.google.com/app/apikey
# https://ai.google.dev/tutorials/setup
# Setting up and saving the API Key for the Gemini language model

# obtaining the api key stored in colab environment

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# configuring genai library with the api key

genai.configure(api_key=GOOGLE_API_KEY)

#### **Creating dictionary of text**

In [ ]:
# Creating dictionary of text and converting to a Pandas DataFrame

Document_1 = {
    "title" : "Llama",
    "content" : "Llama is a collection of pretrained and fine-tuned generative text models ranging in scale between 7 Billion and 70 Billion parameters. Llama's chat models outperform open-source chat models on most benchmarks tested and in human evaluations for helpfulness and safety. Reference : https://huggingface.co/meta-llama "
    }

Document_2 =  {
    "title" : "Mistral AI",
    "content" : "Mistal provides some of the better open source models, outperforming Llama on several benchmarks with 6x faster inference - including the Mixtral 8x7B, high quality sparse mixture of experts model. In particular, the Mixtral-8X7B outperforms or matches GPT-3.5. Reference : https://mistral.ai/news/mixtral-of-experts/"
    }

Document_3 =  {
    "title" : "Abacus.AI",
    "content" : "The Smaug-72B open source model reached an average score of 80 on the Hugging Face LLM leaderboard, including a high GSM8K score for math and reasoning skills. https://twitter.com/abacusai/status/1758174896117420400?ref_src=twsrc%5Egoogle%7Ctwcamp%5Eserp%7Ctwgr%5Etweet "
    }

document = [Document_1, Document_2, Document_3]

df = pd.DataFrame(document)
pd.set_option('display.max_colwidth', None)
df.columns = ['Title', 'Text']

In [ ]:
df

,Title,Text
0,Llama,Llama is a collection of pretrained and fine-tuned generative text models ranging in scale between 7 Billion and 70 Billion parameters. Llama's chat models outperform open-source chat models on most benchmarks tested and in human evaluations for helpfulness and safety. Reference : https://huggingface.co/meta-llama
1,Mistral AI,"Mistal provides some of the better open source models, outperforming Llama on several benchmarks with 6x faster inference - including the Mixtral 8x7B, high quality sparse mixture of experts model. In particular, the Mixtral-8X7B outperforms or matches GPT-3.5. Reference : https://mistral.ai/news/mixtral-of-experts/"
2,Abacus.AI,"The Smaug-72B open source model reached an average score of 80 on the Hugging Face LLM leaderboard, including a high GSM8K score for math and reasoning skills. https://twitter.com/abacusai/status/1758174896117420400?ref_src=twsrc%5Egoogle%7Ctwcamp%5Eserp%7Ctwgr%5Etweet"


#### **Model for creating text embeddings**

- Extract embeddings for each text, which are numeric representations, capturing the semantic meaning behind the text.
- Include the text embeddings as a column in the pandas dataframe.



In [ ]:
# Models for text embedding
for m in genai.list_models():
  if 'embedContent' in m.supported_generation_methods:
    print(m.name)

models/embedding-001


In [ ]:
# Selecting the model for text embeddings
model = 'models/gemini-embedding-001'

def txt_embed(title, text):
  return genai.embed_content(model = model,
                             content = text,
                             task_type = "retrieval_document",
                             title=title)["embedding"]
df['Text_embeddings'] = df.apply(lambda row: txt_embed(row['Title'], row['Text']), axis = 1)
df

,Title,Text,Text_embeddings
0,Llama,Llama is a collection of pretrained and fine-tuned generative text models ranging in scale between 7 Billion and 70 Billion parameters. Llama's chat models outperform open-source chat models on most benchmarks tested and in human evaluations for helpfulness and safety. Reference : https://huggingface.co/meta-llama,"[0.059566624, -0.04107917, -0.036405526, -0.0061017782, 0.04154081, 0.047914322, 0.04911235, -0.02558403, 0.011934878, -0.0036270844, -0.037953723, 0.009089041, -0.0035329047, -0.03279829, 0.0126130255, -0.035014555, -0.0014269049, 0.02844534, -0.00045118423, 0.04765577, 0.027964432, -0.0009260317, -0.02340211, -0.021362212, -0.0095889205, -0.009926082, 0.030851014, -0.04859354, -0.050379366, -0.005153421, -0.0080112675, 0.07239095, 0.0106844995, 0.011521244, 0.005024561, 0.014920746, -0.013912644, -0.0075871255, -0.00018371386, -0.0037125808, 0.016250037, -0.025183279, -0.021587435, 0.008130084, 0.053648766, -0.004974122, 0.018852858, 0.011635348, -0.0017100221, -0.08342639, 0.02539147, 0.016397327, 0.031050619, -0.028444171, -0.009055007, 0.00083381194, -0.03403272, 0.021042485, -0.06209418, 0.008050545, 0.029225364, 0.018031582, 0.0031255095, -0.020708367, -0.009780725, -0.04430215, -0.055010803, 0.03168391, 0.08344177, 0.008972673, -0.0068134186, -0.021773042, 0.03246122, -0.0020149115, -0.04611343, -0.11145333, -0.027392507, 0.017873757, 0.018525276, 0.03256444, 0.04088853, -0.023512451, -0.049415488, -0.02958277, -0.08211645, 0.034967043, -0.050873853, -0.0008175872, -0.085746154, 0.056235455, -0.011419347, -0.031506274, 0.047265045, -0.017877448, 0.023836046, 0.033632312, 0.01515961, -0.023184922, 0.014606004, -0.022389408, ...]"
1,Mistral AI,"Mistal provides some of the better open source models, outperforming Llama on several benchmarks with 6x faster inference - including the Mixtral 8x7B, high quality sparse mixture of experts model. In particular, the Mixtral-8X7B outperforms or matches GPT-3.5. Reference : https://mistral.ai/news/mixtral-of-experts/","[0.022118794, -0.0596182, -0.062297057, -0.011540631, 0.060875654, 0.035907943, 0.026830183, 0.012193753, 0.043668494, 0.010831199, -0.01308296, 0.033765696, 0.015937032, -0.012465031, 0.021861577, -0.05572675, -0.03264504, 0.0078050364, 0.015020257, 0.063498616, 0.055227283, -0.03600914, 0.00793344, -0.029999815, 0.01684297, -0.043431457, 0.0062044547, -0.044130623, -0.057716902, 0.0041836454, -0.05447916, 0.07687413, -0.002468547, -0.0096057905, 0.012465496, -0.020194333, 0.019705001, 0.017833896, 0.01981764, 0.009183083, 0.02508451, -0.03450364, -0.0008310371, -0.038701713, 0.007182404, -0.013190412, 0.039882544, 0.01806635, 0.021531805, -0.07382311, 0.048360083, -0.015289621, 0.056768127, -0.0042967196, 0.028381897, -0.039777633, -0.013018, -0.04040914, 0.015650371, 0.03414326, -0.006825044, -0.01837106, -0.04019478, 0.025004579, -0.043386735, -0.038579606, -0.04576472, -0.0021279713, 0.031828538, -0.017620724, 0.020528482, 0.02767734, 0.07492861, -0.009312688, -0.07293137, -0.13313265, -0.05333327, 0.05547359, 0.0010583231, 0.047710128, 0.026562374, -0.040475834, -0.04599269, -0.038989816, -0.018377488, 0.026160581, -0.022583421, -0.014359532, -0.044272117, 0.060535192, 0.004340757, -0.03953952, 0.066584826, -0.0043259175, -0.013547955, 0.076793335, 0.026794774, -0.050061144, 0.0011422418, -0.06255824, ...]"
2,Abacus.AI,"The Smaug-72B open source model reached an average score of 80 on the Hugging Face LLM leaderboard, including a high GSM8K score for math and reasoning skills. https://twitter.com/abacusai/status/1758174896117420400?ref_src=twsrc%5Egoogle%7Ctwcamp%5Eserp%7Ctwgr%5Etweet","[0.015908912, -0.05579234, -0.06677033, -0.009722288, 0.03613729, 0.021814244, 0.040652245, -0.034669776, 0.03497342, -0.041494288, -0.023222314, 0.006126344, -0.00088149693, -0.032468624, -0.025752183, -0.025944516, 0.015374693, 0.022453314, -0.008085754, 0.02200647, -0.032504093, 0.0021636523, -0.0068475297, -0.018140608, -0.01408

#### **Document search with QA**

- QA system to search these documents based on the text embeddings.
- Embedding of the question will be a vector compared with the vector of the documents using a dot product.
- The dot product represents similarity in direction between the two vectors, which is in between -1 and 1. A  dot product of 1 indicates the vectors are in the same direction, a dot product of 0 indicates the vectors are orthogonal or unrelated and a dot product of -1 indicates the vectors are in opposite direction and are not similar to each other.  

In [ ]:
# Providing the model, query and task-type to the genai.embed_content() method
# for query embedding

query = "How does the Mistral AI large language model compare with Llama model?"

response = genai.embed_content(
    model = model,
    content = query,
    task_type = "retrieval_query"
)


In [ ]:
def find_passage(query, df):
  """
  Computation of distances between the query and each document in the dataframe using
  the dot product.

  """
  query_embed = genai.embed_content(
      model = model,
      content = query,
      task_type = "retrieval_query"
  )

  dot_product = np.dot(np.stack(df['Text_embeddings']), query_embed['embedding'])
  idx = np.argmax(dot_product)
  return df.iloc[idx]['Text']

In [ ]:
# Extracting the model response - relevant passage
text_passage = find_passage(query, df)
text_passage

'Mistal provides some of the better open source models, outperforming Llama on several benchmarks with 6x faster inference - including the Mixtral 8x7B, high quality sparse mixture of experts model. In particular, the Mixtral-8X7B outperforms or matches GPT-3.5. Reference : https://mistral.ai/news/mixtral-of-experts/'

#### **QA Application**

- Using Text generation API for a QA system.

In [ ]:
def make_prompt(query, given_text):
  txt_replace = given_text.replace("'", "").replace('"', "").replace("\n", " ")
  prompt = textwrap.dedent(""" You are a helpful and harmless AI assistant, able to answer \
  questions using text, only from the reference document included below. \
  Please respond in complete sentence and include required backgroud information. \
  If you don't know please say `You don't know`.
  QUESTION : '{query}'

  PASSAGE : '{given_text}'

  ANSWER :

  """
  ).format(query=query, given_text=txt_replace)
  return prompt

query = "What is the Mixtral model? What website url have you been provided with?"

In [ ]:
# Display the prompt for the generative AI model :
prompt = make_prompt(query, text_passage)
print(prompt)

You are a helpful and harmless AI assistant, able to answer   questions using text, only from the reference document included below.   Please respond in complete sentence and include required backgroud information.   If you don't know please say `You don't know`.
 QUESTION : 'What is the Mixtral model? What website url have you been provided with?'

 PASSAGE : 'Mistal provides some of the better open source models, outperforming Llama on several benchmarks with 6x faster inference - including the Mixtral 8x7B, high quality sparse mixture of experts model. In particular, the Mixtral-8X7B outperforms or matches GPT-3.5. Reference : https://mistral.ai/news/mixtral-of-experts/'

 ANSWER :




#### **Gemini-Pro**

- Gemini is a family of multi-model large language models from Google.
- We shall set up 'gemini-pro' model for text generation (based on the prompting technique) in this segment.


In [ ]:
# Setting up the Gemini-pro model using genai.GenerativeModel() method

model=genai.GenerativeModel('gemini-3.1-flash-lite')

In [ ]:
%%time
# response from the model extracted using model.generate_content() method

response = model.generate_content(prompt)

CPU times: user 79.7 ms, sys: 7.83 ms, total: 87.5 ms
Wall time: 3.97 s


In [ ]:
response.candidates[0]

content {
  parts {
    text: "The Mixtral model is a high quality sparse mixture of experts model from Mistral, which outperforms or matches GPT-3.5 on several benchmarks with 6x faster inference. I have been provided with the following website url:  https://mistral.ai/news/mixtral-of-experts/"
  }
  role: "model"
}
finish_reason: STOP
index: 0
safety_ratings {
  category: HARM_CATEGORY_SEXUALLY_EXPLICIT
  probability: NEGLIGIBLE
}
safety_ratings {
  category: HARM_CATEGORY_HATE_SPEECH
  probability: NEGLIGIBLE
}
safety_ratings {
  category: HARM_CATEGORY_HARASSMENT
  probability: NEGLIGIBLE
}
safety_ratings {
  category: HARM_CATEGORY_DANGEROUS_CONTENT
  probability: NEGLIGIBLE
}

In [ ]:
Markdown(response.text)

The Mixtral model is a high quality sparse mixture of experts model from Mistral, which outperforms or matches GPT-3.5 on several benchmarks with 6x faster inference. I have been provided with the following website url:  https://mistral.ai/news/mixtral-of-experts/